In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 3, Finished, Available, Finished, False)

GOLD

In [2]:
#reed tables
df_trips = spark.read.table("trips_silver")
df_zones = spark.read.table("zones_silver")
df_zones.show(5)

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 4, Finished, Available, Finished, False)

+-----------+-------------+--------------------+------------+
|location_id|      borough|                zone|service_zone|
+-----------+-------------+--------------------+------------+
|          1|          EWR|      Newark Airport|         EWR|
|          2|       Queens|         Jamaica Bay|   Boro Zone|
|          3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|          4|    Manhattan|       Alphabet City| Yellow Zone|
|          5|Staten Island|       Arden Heights|   Boro Zone|
+-----------+-------------+--------------------+------------+
only showing top 5 rows



In [3]:
#create dim tables
df_zones = spark.read.table("zones_silver")
df_zones = df_zones.withColumn("location_id", F.col("location_id").cast(IntegerType()))
df_zones.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_zones")

dim_vendor = df_trips.select("vendor_id").distinct()
dim_vendor = dim_vendor.withColumn("vendor_name", 
    F.when(F.col("vendor_id") == 1, "Creative Mobile Technologies")
     .when(F.col("vendor_id") == 2, "Curb Mobility")
     .when(F.col("vendor_id") == 6, "Myle Technologies")
     .when(F.col("vendor_id") == 7, "Helix")
     .otherwise("Unknown"))
dim_vendor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_vendor")

dim_payment = df_trips.select("payment_type").distinct()
dim_payment = dim_payment.withColumn("payment_description",
    F.when(F.col("payment_type") == 0, "Flex Fare")
     .when(F.col("payment_type") == 1, "Credit card")
     .when(F.col("payment_type") == 2, "Cash")
     .when(F.col("payment_type") == 3, "No charge")
     .when(F.col("payment_type") == 4, "Dispute")
     .when(F.col("payment_type") == 5, "Unknown")
     .when(F.col("payment_type") == 6, "Voided trip")
     .otherwise("Unknown"))
dim_payment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_payment_type")

dim_date = df_trips.select(F.to_date("tpep_pickup_datetime").alias("date")).distinct()
dim_date = (dim_date
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
)
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")

dim_ratecode = df_trips.select("ratecode_id").distinct()
dim_ratecode = dim_ratecode.withColumn("ratecode_description",
    F.when(F.col("ratecode_id") == 1, "Standard rate")
     .when(F.col("ratecode_id") == 2, "JFK")
     .when(F.col("ratecode_id") == 3, "Newark")
     .when(F.col("ratecode_id") == 4, "Nassau or Westchester")
     .when(F.col("ratecode_id") == 5, "Negotiated fare")
     .when(F.col("ratecode_id") == 6, "Group ride")
     .otherwise("Unknown"))

dim_ratecode.write.format("delta").mode("overwrite").saveAsTable("gold_dim_ratecode")

#create dim weather

df_weather = spark.read.table("weather_silver")

dim_weather = df_weather.withColumn("date_key", F.date_format("date", "yyyyMMdd").cast(IntegerType()))

dim_weather.write.format("delta").mode("overwrite").saveAsTable("gold_dim_weather")

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 5, Finished, Available, Finished, False)

In [4]:
#view trips table
df_trips.printSchema()

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 6, Finished, Available, Finished, False)

root
 |-- vendor_id: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)



Create Fact Table

In [5]:
#crate date keys
df_trips = df_trips.withColumn("pickup_date_key", F.date_format("tpep_pickup_datetime", "yyyyMMdd").cast(IntegerType()))
df_trips = df_trips.withColumn("dropoff_date_key", F.date_format("tpep_dropoff_datetime", "yyyyMMdd").cast(IntegerType()))

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 7, Finished, Available, Finished, False)

In [6]:
#select columns
fact_trips = df_trips.select(
    "vendor_id",
    "pickup_date_key",
    "dropoff_date_key",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pu_location_id",
    "do_location_id",
    "payment_type",
    "ratecode_id",
    "passenger_count",
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "extra",
    "mta_tax",
    "congestion_surcharge",
    "airport_fee"
)


StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 8, Finished, Available, Finished, False)

In [7]:
fact_trips = fact_trips.withColumn("tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp"))
fact_trips = fact_trips.withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 9, Finished, Available, Finished, False)

In [8]:
#save fact table
fact_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_trips")

StatementMeta(, 3f1e1e97-c8fb-4a6c-8e9c-e5354d18a2aa, 10, Finished, Available, Finished, False)